In [10]:
import os

# set GPU where the code should run
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [11]:
from transformers import pipeline
import torch
import pandas as pd
import re
import gc

In [12]:
#load already sampled dataset
sampled_df = pd.read_json('500_samples_perBucket_data.json', lines=True)

#if you have to sample data, load this data
sampled_df = pd.read_json('poems_complete_with_buckets.json', lines=True)

In [15]:
#extra code if need to clean data for sampling

# filter 1497 and 1913 out of dataset
##df_filtered = sampled_df[~sampled_df['year'].isin([1497, 1913])]

# buckets (without 1497 and 1913)
#buckets = {
    #'1600-1649': (1600, 1649),
    #'1650-1699': (1650, 1699),
    #'1700-1749': (1700, 1749),
    #'1750-1799': (1750, 1799),
    #'1800-1849': (1800, 1849),
    #'1850-1899': (1850, 1899)
#}

# empty dataframe to store samples
#sampled_df = pd.DataFrame()

# samples n examples from each bucket
#for bucket in buckets.keys():
    #bucket_data = df_filtered[df_filtered['bucket'] == bucket]
    #if len(bucket_data) >= 500: ##change number for different sample size
        #sampled_data = bucket_data.sample(n=500) #change number for different sample size
    #else:
        #sampled_data = bucket_data  # if less than n in bucket
    #sampled_df = pd.concat([sampled_df, sampled_data])

# print head of dataset
#print(sampled_df.head())

# store as json
#sampled_df.to_json('500_samples_perBucket_data.json', orient='records', lines=True)

In [16]:
# define prompt templates: 3, 9, 10
templates = [
    {
        "system_prompt": """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind ADJA, ADJD, ADV, APPR, APPRART, APPO, APZR, ART, CARD, FM, ITJ, KOUI, KOUS, KON, KOKOM, NN, NE, PDS, PDAT, PIS, PIAT, PIDAT, PPER,
PPOS, PPOSAT, PRELS, PRELAT, PRF, PWS, PWAT, PWAV, PROAV, PTKZU, PTKNEG, PTKVZ, PTKANT, PTKA, VVFIN, VVIMP, VVINF, VVIZU, VVPP, VAFIN, VAIMP, VAINF,
VAPP, VMFIN, VMINF, VMPP, XY, $, , $(, $. und X. 
X steht für 'andere'.""",
        "prompt": """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Eingabesatz: {tokens}"""
    },
    {
        "system_prompt": """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
""",
        "prompt": """Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes. 
Eingabesatz: {tokens}"""
    },
    {
        "system_prompt": """Sie sind ein deutscher Linguist und Experte für Part-of-Speech Tagging. Sie verwenden STTS (Stuttgart-Tübingen Tagset). 
STTS Tags sind:
ADJA	attributives Adjektiv	[das] große [Haus]
ADJD	adverbiales oder prädikatives Adjektiv	[er fährt] schnell, [er ist] schnell 
ADV	Adverb	schon, bald, doch
APPR	Präposition; Zirkumposition links	in [der Stadt], ohne [mich]
APPRART	Präposition mit Artikel	im [Haus], zur [Sache]
APPO	Postposition	[ihm] zufolge, [der Sache] wegen
APZR	Zirkumposition rechts	[von jetzt] an
ART	bestimmter oder unbestimmter Artikel	der, die, das, ein, eine
CARD	Kardinalzahl	zwei [Männer], [im Jahre] 1994 
FM	Fremdsprachliches Material	[Er hat das mit ``] A big fish ['' übersetzt] 
ITJ	Interjektion	mhm, ach, tja
KOUI	unterordnende Konjunktion mit ``zu'' und Infinitiv	um [zu leben], anstatt [zu fragen]
KOUS	unterordnende Konjunktion mit Satz	weil, dass, damit, wenn, ob
KON	nebenordnende Konjunktion	und, oder, aber
KOKOM	Vergleichskonjunktion	als, wie
NN	normales Nomen	Tisch, Herr, [das] Reisen
NE	Eigennamen	Hans, Hamburg, HSV
PDS	substituierendes Demonstrativpronomen	dieser, jener
PDAT	attribuierendes Demonstrativpronomen	jener [Mensch]
PIS	substituierendes Indefinitpronomen	keiner, viele, man, niemand
PIAT	attribuierendes Indefinitpronomen ohne Determiner	kein [Mensch], irgendein [Glas]
PIDAT	attribuierendes Indefinitpronomen mit Determiner	[ein] wenig [Wasser], [die] beiden [Brüder] 
PPER	irreflexives Personalpronomen	ich, er, ihm, mich, dir
PPOSS	substituierendes Possessivpronomen	meins, deiner
PPOSAT	attribuierendes Possessivpronomen	mein [Buch], deine [Mutter]
PRELS	substituierendes Relativpronomen	[der Hund ,] der
PRELAT	attribuierendes Relativpronomen	[der Mann ,] dessen [Hund]
PRF	reflexives Personalpronomen	sich, einander, dich, mir
PWS	substituierendes Interrogativpronomen	wer, was
PWAT	attribuierendes Interrogativpronomen	welche[Farbe], wessen [Hut]
PWAV	adverbiales Interrogativ- oder Relativpronomen	warum, wo, wann, worüber, wobei
PAV	Pronominaladverb	dafür, dabei, deswegen, trotzdem
PTKZU	``zu'' vor Infinitiv	zu [gehen]
PTKNEG	Negationspartikel	nicht
PTKVZ	abgetrennter Verbzusatz	[er kommt] an, [er fährt] rad
PTKANT	Antwortpartikel	ja, nein, danke, bitte
PTKA	Partikel bei Adjektiv oder Adverb	am [schönsten], zu [schnell] 
TRUNC	Kompositions-Erstglied	An- [und Abreise]
VVFIN	finites Verb, voll	[du] gehst, [wir] kommen [an]
VVIMP	Imperativ, voll	komm [!]
VVINF	Infinitiv, voll	gehen, ankommen
VVIZU	Infinitiv mit ``zu'', voll	anzukommen, loszulassen
VVPP	Partizip Perfekt, voll	gegangen, angekommen
VAFIN	finites Verb, aux	[du] bist, [wir] werden
VAIMP	Imperativ, aux	sei [ruhig !]
VAINF	Infinitiv, aux	werden, sein
VAPP	Partizip Perfekt, aux	gewesen
VMFIN	finites Verb, modal	dürfen
VMINF	Infinitiv, modal	wollen
VMPP	Partizip Perfekt, modal	gekonnt, [er hat gehen] können
XY	Nichtwort, Sonderzeichen enthaltend	3:7, H2O, D2XW3 
$,	Komma	,
$.	Satzbeendende Interpunktion	. ? ! ; :
$(	sonstige Satzzeichen; satzintern	- [,]()
X	steht für 'andere'	
""",
        "prompt": """"Bitte geben Sie für jedes Wort im Eingabesatz die POS-Tags an. 
Die Eingabe besteht aus einer Liste von Wörtern im Satz. 
Das Ausgabeformat sollte eine Liste von Tupeln sein, wobei jedes Tupel aus einem Wort aus dem Eingabetext und der dazugehörigen POS-Markierung besteht. 
Ihre Antwort sollte nur eine Liste von Tupeln enthalten, in der Reihenfolge, in der die Wörter im Eingabesatz erscheinen, 
wobei jedes Tupel die entsprechende POS-Markierung für ein Wort enthält. 
Bitte geben Sie nur die Liste und nichts anderes zurück. 
Hier sind ein paar Beispiele, was Sie machen sollen:
Eingabesatz: ['Balde', 'rings', 'die', 'Sterne', 'bleichen', 'Und', 'ermattet', 'von', 'Beschwerde', 'Wächsern', 'ihre', 'Wangen', 'bleichen', '.', 'Fäulnis', 'wittert', 'aus', 'der', 'Erde', '.']
Ausgabe: [('Balde','NN'),('rings','ADV'),('die','ART'),('Sterne','NN'),('bleichen','VVINF'),('Und','KON'),('ermattet','VVFIN'),('von','APPR'),('Beschwerde','NN'),('Wächsern','KOUS'),('ihre','PPOSAT'),('Wangen','NN'),('bleichen','VVINF'),('.','$.'),('Fäulnis','NN'),('wittert','VVFIN'),('aus','APPR'),('der','ART'),('Erde','NN'),('.','$.')]

Eingabesatz: ['Traurig', 'rauscht', 'das', 'Rohr', 'im', 'Tümpel', 'Und', 'sie', 'friert', 'in', 'sich', 'gekauert', '.', 'Fern', 'ein', 'Hahn', 'kräht', '.', 'Übern', 'Tümpel', 'Hart', 'und', 'grau', 'der', 'Morgen', 'schauert', '.']
Ausgabe: [('Traurig','ADJD'), ('rauscht','VVFIN'), ('das','ART'), ('Rohr','NN'), ('im','APPRART'), ('Tümpel','NN'), ('Und','KON'), ('sie','PPER'), ('friert','VVFIN'), ('in','APPR'), ('sich','PRF'), ('gekauert','VVPP'), ('.','$.'), ('Fern','NN'),('ein','ART'), ('Hahn','NN'), ('kräht','VVFIN'), ('.','$.'), ('Übern','NN'), ('Tümpel','NN'), ('Hart','NE'), ('und','KON'), ('grau','ADJD'), ('der','ART'), ('Morgen','NN'), ('schauert','VVFIN'), ('.','$.')]

Eingabesatz: ['In', 'der', 'Schmiede', 'dröhnt', 'der', 'Hammer', 'Und', 'sie', 'huscht', 'am', 'Tor', 'vorüber', '.', 'Glührot', 'schwingt', 'der', 'Knecht', 'den', 'Hammer', 'Und', 'sie', 'schaut', 'wie', 'tot', 'hinüber', '.']
Ausgabe: [('In','APPR'), ('der','ART'), ('Schmiede','NN'), ('dröhnt','VVFIN'), ('der','ART'), ('Hammer','NN'), ('Und','KON'), ('sie','PPER'), ('huscht','VVFIN'), ('am','APPART'), ('Tor','NN'), ('vorüber','PTKVZ'), ('.','$.'), ('Glührot','NN'), ('schwingt','VVFIN'), ('der','ART'), ('Knecht','NN'), ('den','ART'), ('Hammer','NN'), ('Und','KON'), ('sie','PPER'), ('schaut','VVFIN'), ('wie','KOKOM'), ('tot','ADJD'), ('hinüber','ADV'), ('.','$.')]
Eingabesatz: {tokens}"""
    }
]

# define prompt-format
prompt_format = """<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"""

# function to format prompts
def format_prompt(system_prompt, prompt):
    return prompt_format.format(system_prompt=system_prompt, prompt=prompt)
#list to store prompt templates
prompt_templates = []
# format templates
for template in templates:
    formatted_prompt = format_prompt(template['system_prompt'], template['prompt'])
    prompt_templates.append(formatted_prompt)

In [ ]:
#leo-mistral-hessianai-7b-chat
generator = pipeline(model="LeoLM/leo-mistral-hessianai-7b-chat", device="cuda", torch_dtype=torch.float16) # True for flash-attn2 else False

In [ ]:
# list to store results
results = []
id = 0
# POS-Tagging for every text in the three Prompt-Templates
for idx, row in sampled_df.iterrows():
    text = row['tokens']
    bucket = row['bucket']
    labels = row['labels']
    
    # for every Prompt-Template
    for i, prompt in enumerate(prompt_templates):
        prompt_input = prompt.format(tokens=text)
        # POS-Tagging ausführen
        pos_result = generator(prompt_input, do_sample=True, top_p=0.95, max_new_tokens=1000, return_full_text=False)
        
        #extract only the assistant's answer
        response_text = pos_result[0]['generated_text']
        
        
        id = id+1
        
        # store result
        results.append({
            'year': row['year'],
            'bucket': bucket,
            'labels': labels,
            'template_id': i + 1, #1 for template3, 2 for template9 and 3 for template10
            'POS_Result': response_text
        })
        
    # check in which bucket we are, number of samples * 3
    if id % 1500 == 0:
        print("finish bucket")
        
        
# store results in dataframe
df_results = pd.DataFrame(results)

# store as json
df_results.to_json('500_samples_perBucket_RESULTS_bucket5.json', orient='records', lines=True)

In [19]:
# function to extract POS-Tags with Regex
def extract_pos_tags(text):
    return re.findall(r"'([A-Z$]+(?:[A-Z.,(]+)?)'", text)

#apply regex
df_results['extracted_pos_tags'] = df_results['POS_Result'].apply(extract_pos_tags)

# store as json
df_results.to_json('500_samples_perBucket_RESULTS_extracted_bucket5.json', orient='records', lines=True)

In [ ]:
#calculate average accuracy
def calculate_accuracy(pos_tags, predicted_tags):
    correct = sum(1 for pos, pred in zip(pos_tags, predicted_tags) if pos == pred)
    return correct / len(pos_tags) *100

average_accuracies = []
for bucket in df_results['bucket'].unique():
    for template_num in df_results['template_id'].unique():
        bucket_template_data = df_results[(df_results['bucket'] == bucket) & (df_results['template_id'] == template_num)]
        
        # calculate average accuracy for this bucket and this template
        accuracies = []
        for idx, row in bucket_template_data.iterrows():
            extracted_tags = row['extracted_pos_tags']
            true_tags = row['labels'] 
            accuracy = calculate_accuracy(true_tags, extracted_tags)
            accuracies.append(accuracy)
        
        average_accuracy = sum(accuracies) / len(accuracies)  if accuracies else 0
        
        average_accuracies.append({
            'bucket': bucket,
            'template': template_num,
            'average_accuracy': average_accuracy
        })

# store accuracy results as DataFrame
df_accuracy = pd.DataFrame(average_accuracies)       
# print head of dataframe
print(df_accuracy.head())

# store accuracy results as JSON
df_accuracy.to_json('accuracy_perBucket_perTemplate_500samples_bucket5.json', orient='records', lines=True)

In [ ]:
# print finish code
print("Script finished successfully!")

# finish script
import os
os.system("exit")